In [1]:
import csv
import re
from pathlib import Path

import pandas as pd
from tqdm import tqdm

In [2]:
RAW_CMHC_ROOT   = Path("../../data/cmhc-rental-raw")
CLEAN_CMHC_ROOT = Path("../../data/cmhc-rental-clean")

RAW_STATSCAN_ROOT   = Path("../../data/statscan-quarterly-rental-raw")
CLEAN_STATSCAN_ROOT = Path("../../data/statscan-quarterly-rental-clean")

# Column headers for cleaned CMHC bedroom-type files
CMHC_BEDROOM_COLS = ["Year", "Studio", "1 Bedroom", "2 Bedroom", "3 Bedroom+", "Total"]

# Column headers for cleaned CMHC primary-summary files
CMHC_SUMMARY_COLS = ["Year", "Vacancy Rate", "Availability Rate", "Average Rent", "Median Rent", "Pct Change", "Units"]

# StatsCan raw unit-type labels -> clean column names
STATSCAN_UNIT_MAP = {
    "House - 3 or more bedrooms": "House 3+br",
    "Apartment - 1 bedroom":      "Apt 1br",
    "Apartment - 2 bedrooms":     "Apt 2br",
    "Room":                       "Room",
}

## Section 1 — CMHC Rental Market Data

Raw files come from the CMHC Housing Market Information Portal and live under
`data/cmhc-rental-raw/<group>/<subfolder>/<city>.csv`.  There are three groups:

| Group | Scope | Subfolders |
|---|---|---|
| `on-csd-primary` | Ontario Census Subdivisions (cities) | `avg-rent`, `pct-change-rent`, `vacancy-rate`, `primary-summary` |
| `can-cma-primary` | Canadian CMAs | `primary-summary` |
| `can-pr-primary` | Canadian provinces | `primary-summary` |

**Bedroom-type subfolders** (`avg-rent`, `pct-change-rent`, `vacancy-rate`) share the
same format: each row is a year (October survey) and every value column is interleaved
with a quality-code column that we discard.  

**Primary-summary subfolders** contain aggregate (total-only) statistics across six
metrics: vacancy rate, availability rate, average rent, median rent, % change, and
unit count.

Clean outputs mirror the raw directory tree under `data/cmhc-rental-clean/`.

In [3]:
# Matches "YYYY October" row labels (the annual survey date)
YEAR_RE = re.compile(r"^(\d{4})\s+October\b", re.IGNORECASE)

def _row_value(row: list, index: int) -> str:
    return row[index] if index < len(row) else ""

def _parse_value(raw: str, is_float: bool):
    """Convert a raw CMHC cell to a Python number, or None if absent.

    Special CMHC tokens:
      **  -> data not available  -> None
      ++  -> change < 0.05 %     -> 0.0 / 0
    """
    value = raw.strip() if raw else ""
    if not value or value == "**":
        return None
    if value == "++":
        return 0.0 if is_float else 0
    value = value.replace(",", "")
    try:
        num = float(value)
    except ValueError:
        return None
    if is_float:
        return round(num, 2)
    return int(round(num))

### 1a — Bedroom-type subfolders (`avg-rent`, `pct-change-rent`, `vacancy-rate`)

Raw layout (after two title rows and one header row):
```
1998 October,  588, a,  727, a,  882, a, ...
               ^^^  ^^  ^^^  ^^           <- value / quality-code pairs
```
We extract every other column starting at index 1 (Studio, 1 Bedroom, 2 Bedroom,
3 Bedroom+, Total) and span the full observed year range, filling gaps with NaN.

In [4]:
def parse_cmhc_bedroom_csv(file_path: Path, is_float: bool) -> pd.DataFrame:
    with file_path.open("r", encoding="utf-8", errors="replace", newline="") as fh:
        rows = list(csv.reader(fh))

    year_map: dict[int, list] = {}
    for row in rows:
        if not row:
            continue
        match = YEAR_RE.match(row[0].strip())
        if not match:
            continue
        year = int(match.group(1))
        year_map[year] = [
            year,
            _parse_value(_row_value(row, 1), is_float),   # Studio
            _parse_value(_row_value(row, 3), is_float),   # 1 Bedroom
            _parse_value(_row_value(row, 5), is_float),   # 2 Bedroom
            _parse_value(_row_value(row, 7), is_float),   # 3 Bedroom+
            _parse_value(_row_value(row, 9), is_float),   # Total
        ]

    if not year_map:
        return pd.DataFrame(columns=CMHC_BEDROOM_COLS)

    min_yr, max_yr = min(year_map), max(year_map)
    data = [year_map.get(y, [y, None, None, None, None, None]) for y in range(min_yr, max_yr + 1)]
    return pd.DataFrame(data, columns=CMHC_BEDROOM_COLS)


def clean_cmhc_bedroom_subfolder(src_dir: Path, dst_dir: Path, is_float: bool) -> None:
    dst_dir.mkdir(parents=True, exist_ok=True)
    files = sorted(src_dir.glob("*.csv"))
    for fp in tqdm(files, desc=f"  {src_dir.name}"):
        df = parse_cmhc_bedroom_csv(fp, is_float)
        out = dst_dir / fp.name
        if is_float:
            for col in CMHC_BEDROOM_COLS[1:]:
                df[col] = pd.to_numeric(df[col], errors="coerce")
            df.to_csv(out, index=False, na_rep="", float_format="%.1f")
        else:
            for col in CMHC_BEDROOM_COLS[1:]:
                df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
            df.to_csv(out, index=False, na_rep="")

### 1b — Primary-summary subfolders

These files have a single combined table with six metrics (all for the **Total** bedroom
category) rather than a bedroom breakdown.  Raw layout after two title rows:
```
header:  , Vacancy Rate (%), , Availability Rate (%), , Average Rent ($), , Median Rent ($), , % Change, , Units,
data:    1998 October, 0.9, a, **, , 808, a, 775, a, 6.7, a, "254,774",
```
Column indices (0-based): 1=vacancy, 3=availability, 5=avg rent, 7=median rent,
9=pct change, 11=units.  Availability is often `**` (unreported before ~2005).

In [5]:
def parse_cmhc_summary_csv(file_path: Path) -> pd.DataFrame:
    with file_path.open("r", encoding="utf-8", errors="replace", newline="") as fh:
        rows = list(csv.reader(fh))

    year_map: dict[int, list] = {}
    for row in rows:
        if not row:
            continue
        match = YEAR_RE.match(row[0].strip())
        if not match:
            continue
        year = int(match.group(1))
        year_map[year] = [
            year,
            _parse_value(_row_value(row, 1),  is_float=True),   # Vacancy Rate
            _parse_value(_row_value(row, 3),  is_float=True),   # Availability Rate
            _parse_value(_row_value(row, 5),  is_float=False),  # Average Rent
            _parse_value(_row_value(row, 7),  is_float=False),  # Median Rent
            _parse_value(_row_value(row, 9),  is_float=True),   # Pct Change
            _parse_value(_row_value(row, 11), is_float=False),  # Units
        ]

    if not year_map:
        return pd.DataFrame(columns=CMHC_SUMMARY_COLS)

    min_yr, max_yr = min(year_map), max(year_map)
    data = [year_map.get(y, [y, None, None, None, None, None, None]) for y in range(min_yr, max_yr + 1)]
    return pd.DataFrame(data, columns=CMHC_SUMMARY_COLS)


def clean_cmhc_summary_subfolder(src_dir: Path, dst_dir: Path) -> None:
    dst_dir.mkdir(parents=True, exist_ok=True)
    files = sorted(src_dir.glob("*.csv"))
    for fp in tqdm(files, desc=f"  {src_dir.name}"):
        df = parse_cmhc_summary_csv(fp)
        # Float columns
        for col in ["Vacancy Rate", "Availability Rate", "Pct Change"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        # Integer columns
        for col in ["Average Rent", "Median Rent", "Units"]:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
        out = dst_dir / fp.name
        df.to_csv(out, index=False, na_rep="", float_format="%.2f")

### 1c — Run CMHC Cleaning

Iterates every group directory under `cmhc-rental-raw/`, dispatches each subfolder
to the appropriate parser (bedroom-type vs. summary), and writes results to the
mirrored path under `cmhc-rental-clean/`.

In [6]:
# Subfolders that contain bedroom-breakdown data; everything else is primary-summary.
BEDROOM_SUBFOLDERS = {"avg-rent", "pct-change-rent", "vacancy-rate"}
# Subfolders where values are percentages/rates (float) rather than dollar amounts (int).
FLOAT_SUBFOLDERS   = {"pct-change-rent", "vacancy-rate"}

for group_dir in sorted(RAW_CMHC_ROOT.iterdir()):
    if not group_dir.is_dir():
        continue
    print(f"\n{group_dir.name}")
    for subfolder_dir in sorted(group_dir.iterdir()):
        if not subfolder_dir.is_dir():
            continue
        dst_dir = CLEAN_CMHC_ROOT / group_dir.name / subfolder_dir.name
        if subfolder_dir.name in BEDROOM_SUBFOLDERS:
            is_float = subfolder_dir.name in FLOAT_SUBFOLDERS
            clean_cmhc_bedroom_subfolder(subfolder_dir, dst_dir, is_float)
        else:
            clean_cmhc_summary_subfolder(subfolder_dir, dst_dir)


can-cma-primary



  primary-summary:   0%|          | 0/42 [00:00<?, ?it/s]


  primary-summary:  45%|████▌     | 19/42 [00:00<00:00, 181.54it/s]


  primary-summary:  90%|█████████ | 38/42 [00:00<00:00, 176.79it/s]


  primary-summary: 100%|██████████| 42/42 [00:00<00:00, 175.99it/s]

...


  primary-summary:  85%|████████▌ | 29/34 [00:00<00:00, 140.54it/s]


  primary-summary: 100%|██████████| 34/34 [00:00<00:00, 138.33it/s]


  vacancy-rate:   0%|          | 0/34 [00:00<?, ?it/s]


  vacancy-rate:  62%|██████▏   | 21/34 [00:00<00:00, 202.73it/s]


  vacancy-rate: 100%|██████████| 34/34 [00:00<00:00, 197.40it/s]

## Section 2 — StatsCan Quarterly Asking Rents (Table 46-10-0092-01)

Source: `statscan-quarterly-rental-raw/can-cma-all/4610009201-noSymbol.csv`

The raw file is a wide table where:
- Columns = quarters (`Q1 2019` … `Q1 2026`)
- Row groups = one CMA per group, with a sub-row for each rental unit type
  (`House - 3 or more bedrooms`, `Apartment - 1 bedroom`, `Apartment - 2 bedrooms`, `Room`)
- Values = average asking rent in dollars; `..` = suppressed/unavailable, `F` = too unreliable to publish

Clean output: one CSV per CMA under `statscan-quarterly-rental-clean/can-cma-all/`,
with quarters as rows and unit types as columns — the same orientation as the CMHC
bedroom files (time as the index axis).

CMA names are stripped of their `, Census metropolitan area (CMA)` suffix and
lower-cased to match the naming convention used for CMHC files.

In [7]:
# Rows that appear in the statscan CSV but are header/metadata, not data.
_STATSCAN_SKIP = {"", "Estimates", "Geography", "Rental unit type", "Dollars"}

# Only rows whose geography field matches this pattern are real CMA entries.
_GEO_RE = re.compile(r"Census metropolitan area", re.IGNORECASE)

# Special-case overrides applied before generic normalisation.
_GEO_OVERRIDES = {
    "Ottawa - Gatineau (Ontario part), Census metropolitan area (CMA)": "ottawa",
    "Ottawa - Gatineau (Quebec part), Census metropolitan area (CMA)":  "gatineau",
}


def geo_to_filename(geo: str) -> str:
    """Convert a raw StatsCan geography label to a clean filename stem.

    Rules (applied in order):
      1. Hard-coded overrides for split CMAs (Ottawa/Gatineau).
      2. Strip ', Census metropolitan area (CMA)' suffix.
      3. Replace accented e variants with plain e.
      4. Collapse ' - ' (spaced hyphen) to '-'.
      5. Lower-case.
    """
    if geo in _GEO_OVERRIDES:
        return _GEO_OVERRIDES[geo]
    name = re.sub(r",?\s*Census metropolitan area.*$", "", geo, flags=re.IGNORECASE).strip()
    name = re.sub(r"[éèêë]", "e", name)
    name = re.sub(r" - ", "-", name)
    return name.lower()


def _is_real_geo(geo: str) -> bool:
    """Reject footnote rows (single chars, numbers, '..') that appear at file bottom."""
    return geo == "All census metropolitan areas" or bool(_GEO_RE.search(geo))


def parse_statscan_quarterly(file_path: Path) -> dict[str, pd.DataFrame]:
    """Parse the StatsCan quarterly asking-rent wide table.

    Returns a mapping of raw geography label -> DataFrame with columns
    [Quarter, House 3+br, Apt 1br, Apt 2br, Room].
    """
    with file_path.open("r", encoding="utf-8-sig", errors="replace", newline="") as fh:
        rows = list(csv.reader(fh))

    # Locate the quarters header row (first row whose 3rd column starts with "Q")
    quarters: list[str] = []
    data_start = 0
    for i, row in enumerate(rows):
        if len(row) > 2 and row[2].strip().startswith("Q"):
            quarters = [c.strip() for c in row[2:] if c.strip()]
            data_start = i + 1
            break

    if not quarters:
        raise ValueError(f"Could not locate quarters header in {file_path}")

    n = len(quarters)
    cities: dict[str, dict[str, list]] = {}
    current_geo: str | None = None

    for row in rows[data_start:]:
        if not row or not any(c.strip() for c in row):
            continue

        geo  = row[0].strip().strip('"') if row else ""
        unit = row[1].strip().strip('"') if len(row) > 1 else ""

        if geo in _STATSCAN_SKIP and unit in _STATSCAN_SKIP:
            continue
        if unit in _STATSCAN_SKIP:
            continue

        if geo and _is_real_geo(geo):
            current_geo = geo
            if current_geo not in cities:
                cities[current_geo] = {}

        if unit and current_geo and unit in STATSCAN_UNIT_MAP:
            values: list = []
            for raw in row[2:2 + n]:
                v = raw.strip().strip('"').replace(",", "")
                if v in ("..", "F", ""):
                    values.append(None)
                else:
                    try:
                        values.append(int(round(float(v))))
                    except ValueError:
                        values.append(None)
            values.extend([None] * (n - len(values)))
            cities[current_geo][unit] = values[:n]

    result: dict[str, pd.DataFrame] = {}
    for geo, unit_data in cities.items():
        df = pd.DataFrame({"Quarter": quarters})
        for raw_unit, clean_col in STATSCAN_UNIT_MAP.items():
            df[clean_col] = pd.array(
                unit_data.get(raw_unit, [None] * n), dtype="Int64"
            )
        result[geo] = df

    return result


def clean_statscan_quarterly(src_file: Path, dst_dir: Path) -> None:
    # Remove any stale files from prior runs before writing fresh output.
    if dst_dir.exists():
        for f in dst_dir.glob("*.csv"):
            f.unlink()
    dst_dir.mkdir(parents=True, exist_ok=True)
    city_frames = parse_statscan_quarterly(src_file)
    for geo, df in tqdm(city_frames.items(), desc="  can-cma-all"):
        filename = geo_to_filename(geo) + ".csv"
        df.to_csv(dst_dir / filename, index=False, na_rep="")

In [8]:
statscan_src = RAW_STATSCAN_ROOT / "can-cma-all" / "4610009201-noSymbol.csv"
statscan_dst = CLEAN_STATSCAN_ROOT / "can-cma-all"

print("StatsCan quarterly asking rents")
clean_statscan_quarterly(statscan_src, statscan_dst)
print(f"  → wrote {len(list(statscan_dst.glob('*.csv')))} city files to {statscan_dst}")

StatsCan quarterly asking rents



  can-cma-all:   0%|          | 0/43 [00:00<?, ?it/s]


  can-cma-all: 100%|██████████| 43/43 [00:00<00:00, 584.67it/s]

  → wrote 43 city files to ../../data/statscan-quarterly-rental-clean/can-cma-all
